# Example use of UTRfx package to process uORFs

This is a document showing how to use UTRfx to identify upstream open reading frames (uORFs) and extract some features of interest.

## Usage

### Pre-requisites

##### GTF file

To start with, a GTF file is needed to create transcripts with the 5'UTR coordinates. This is the only necessary file.          
There can be two options:

- The GTF file does not indicate explicitly the 5'UTR region as seen here:

In [12]:
import os

file_path_no_explicit = os.path.join("..", "..", "tests", "data", "Homo.sapiens.GRCh38_sample_chr22.gtf")

with open(file=file_path_no_explicit, mode="r", encoding='utf-8') as file:
    lines = file.readlines()
    for line in lines[160:169]:
        print(line.strip())


chr22	HAVANA	CDS	44732251	44732391	.	+	0	gene_id "ENSG00000186654.22"; transcript_id "ENST00000432186.6"; gene_type "protein_coding"; gene_name "PRR5"; transcript_type "protein_coding"; transcript_name "PRR5-206"; exon_number 7; exon_id "ENSE00003658491.1"; level 2; protein_id "ENSP00000400925.2"; transcript_support_level "2"; hgnc_id "HGNC:31682"; tag "alternative_5_UTR"; tag "basic"; tag "appris_alternative_1"; tag "CCDS"; ccdsid "CCDS14059.1"; havana_gene "OTTHUMG00000150460.6"; havana_transcript "OTTHUMT00000318206.2";
chr22	HAVANA	exon	44735027	44735162	.	+	.	gene_id "ENSG00000186654.22"; transcript_id "ENST00000432186.6"; gene_type "protein_coding"; gene_name "PRR5"; transcript_type "protein_coding"; transcript_name "PRR5-206"; exon_number 8; exon_id "ENSE00003692865.1"; level 2; protein_id "ENSP00000400925.2"; transcript_support_level "2"; hgnc_id "HGNC:31682"; tag "alternative_5_UTR"; tag "basic"; tag "appris_alternative_1"; tag "CCDS"; ccdsid "CCDS14059.1"; havana_gene "OTTHUM

As it can be seen, the transcript `ENST00000432186.6` has three UTR regions, but it is not indicated which one/s correspond to the 5'UTR region. Therefore, the `extract_fice_utrs_if_not_explicit` method of `GTFio` class should be used in this case.

- The GTF file does indicate explicitly the 5'UTR region as seen here:

In [13]:
import os

file_path_explicit = os.path.join("..", "..", "tests", "data", "Homo.sapiens.GRCh38_sample_chr8.gtf")

with open(file=file_path_explicit, mode="r", encoding='cp1252') as file:
    lines = file.readlines()
    for line in lines[170:181]:
        print(line.strip())

8	ensembl_havana	exon	19955841	19956083	.	+	.	gene_id "ENSG00000175445"; gene_version "17"; transcript_id "ENST00000650287"; transcript_version "1"; exon_number "6"; gene_name "LPL"; gene_source "ensembl_havana"; gene_biotype "protein_coding"; transcript_name "LPL-207"; transcript_source "ensembl_havana"; transcript_biotype "protein_coding"; tag "CCDS"; ccds_id "CCDS6012"; exon_id "ENSE00001206552"; exon_version "1"; tag "gencode_basic"; tag "gencode_primary"; tag "MANE_Select"; tag "Ensembl_canonical";
8	ensembl_havana	CDS	19955841	19956083	.	+	2	gene_id "ENSG00000175445"; gene_version "17"; transcript_id "ENST00000650287"; transcript_version "1"; exon_number "6"; gene_name "LPL"; gene_source "ensembl_havana"; gene_biotype "protein_coding"; transcript_name "LPL-207"; transcript_source "ensembl_havana"; transcript_biotype "protein_coding"; tag "CCDS"; ccds_id "CCDS6012"; protein_id "ENSP00000497642"; protein_version "1"; tag "gencode_basic"; tag "gencode_primary"; tag "MANE_Select"; ta

As it can be seen, the transcript `ENST00000524240` has a 5'UTR region (marked as `five_prime_utr`) and a 3'UTR (marked as `three_prime_utr`). Therefore, the `extract_fice_utrs_if_explicit` method of `GTFio` class should be used in this case.

##### Variants file

In this example, we will use a file with pathogenic variants of a gene, we recommend following a VCF-style or an actual VCF file. These variants are extremely uncommon, that is the reason why this file is needed in this case. This is not necessary, single transcripts can be processed.                          
We will be using an Excel file converted into a pandas `DataFrame`:


In [14]:
import os
import pandas as pd

file_path_variants = os.path.join("..", "..", "tests", "data", "Variants_gene_HR.xlsx")

variants_df = pd.read_excel(file_path_variants)
variants_df.columns = ["chr", "start", "end", "ref", "alt", "gene_symbol"]
print(variants_df)

    chr     start       end ref alt gene_symbol
0     8  22130605  22130605   T   C          HR
1     8  22130606  22130606   A   G          HR
2     8  22130627  22130627   C   G          HR
3     8  22130633  22130633   C   T          HR
4     8  22130635  22130635   G   A          HR
5     8  22130636  22130636   G   C          HR
6     8  22130638  22130638   A   T          HR
7     8  22130689  22130689   G   T          HR
8     8  22130702  22130702   G   A          HR
9     8  22130706  22130706   C   T          HR
10    8  22130707  22130707   A   T          HR
11    8  22130707  22130707   A   G          HR
12    8  22130708  22130708   T   C          HR


There are 13 variants of the gene HR, which is located in the chromosome 8.

##### VCF file

This VCF file was downloaded from GnomAD, which contains variants within a small region of the chromosome 8, corresponding to the second uORF of the HR gene.

In [15]:
import os
import gzip

vcf_fpath = os.path.join("..", "..", "tests", "data", "gnomad.genomes.v4.1.sites.chr8.sample.vcf.gz")

with gzip.open(vcf_fpath, mode="rt") as file:
    lines = file.readlines()
    for line in lines[285:]:
        print(line)

#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO

chr8	22130611	rs1039104495	C	T	.	PASS	AC=2;AN=152148;AF=1.31451e-05;grpmax=nfe;fafmax_faf95_max=4.88e-06;fafmax_faf95_max_gen_anc=nfe;AC_XX=2;AF_XX=2.56977e-05;AN_XX=77828;nhomalt_XX=0;AC_XY=0;AF_XY=0;AN_XY=74320;nhomalt_XY=0;nhomalt=0;AC_afr_XX=0;AF_afr_XX=0;AN_afr_XX=22162;nhomalt_afr_XX=0;AC_afr_XY=0;AF_afr_XY=0;AN_afr_XY=19294;nhomalt_afr_XY=0;AC_afr=0;AF_afr=0;AN_afr=41456;nhomalt_afr=0;AC_ami_XX=0;AF_ami_XX=0;AN_ami_XX=468;nhomalt_ami_XX=0;AC_ami_XY=0;AF_ami_XY=0;AN_ami_XY=442;nhomalt_ami_XY=0;AC_ami=0;AF_ami=0;AN_ami=910;nhomalt_ami=0;AC_amr_XX=0;AF_amr_XX=0;AN_amr_XX=6790;nhomalt_amr_XX=0;AC_amr_XY=0;AF_amr_XY=0;AN_amr_XY=8492;nhomalt_amr_XY=0;AC_amr=0;AF_amr=0;AN_amr=15282;nhomalt_amr=0;AC_asj_XX=0;AF_asj_XX=0;AN_asj_XX=1868;nhomalt_asj_XX=0;AC_asj_XY=0;AF_asj_XY=0;AN_asj_XY=1604;nhomalt_asj_XY=0;AC_asj=0;AF_asj=0;AN_asj=3472;nhomalt_asj=0;AC_eas_XX=0;AF_eas_XX=0;AN_eas_XX=2272;nhomalt_eas_XX=0;AC_eas_XY=0;AF_eas_XY=0;AN_eas_XY=2898;nhom

### Workflow

Independently of which GTF file type, the first step is common for both: conversion into a pandas `DataFrame`. This method is shown in this example, but it is automatically done because `__init__` method calls the `gtf_to_dataframe()` method.               
This notebook example will continue with the second GTF file type (5'UTR regions explicitly indicated), in our case, it's only the chromosome 8 genes.                 
The file is opened, the conversion is done, a new column is created with the `transcript_id` field of the `attibutes` column and then, this last column and others are eliminated.

In [16]:
from utrfx.gtf_io import GTFio

gtf_df = GTFio(fpath=file_path_explicit).gtf_to_dataframe()
gtf_df.head()


,seqname,feature,start,end,strand,transcript_id
0,8,transcript,22713251,22927914,-,ENST00000256404
1,8,exon,22927823,22927914,-,ENST00000256404
2,8,exon,22927584,22927720,-,ENST00000256404
3,8,CDS,22927584,22927714,-,ENST00000256404
4,8,start_codon,22927712,22927714,-,ENST00000256404


Once the `DataFrame` is created, each 5'UTR region is transformed into a `TranscriptCoordinates` with one of the two methods, as explained before.

In [17]:
from utrfx.gtf_io import GTFio
from utrfx.genome import GRCh38

gtf_file = GTFio(fpath=file_path_explicit)
transcripts = gtf_file.extract_five_utrs_if_explicit(genome_build=GRCh38)
print(transcripts)

[TranscriptCoordinates(tx_id=ENST00000256404, five_utr=FiveUTRCoordinates(regions=2 regions: (chr8, 122210722, 122210814, -), (chr8, 122210916, 122210922, -))), TranscriptCoordinates(tx_id=ENST00000289734, five_utr=FiveUTRCoordinates(regions=1 regions: (chr8, 103341014, 103341098, -))), TranscriptCoordinates(tx_id=ENST00000381418, five_utr=FiveUTRCoordinates(regions=2 regions: (chr8, 123007626, 123008209, -), (chr8, 123009426, 123009466, -))), TranscriptCoordinates(tx_id=ENST00000517969, five_utr=FiveUTRCoordinates(regions=2 regions: (chr8, 42154130, 42154536, +), (chr8, 42154615, 42154687, +))), TranscriptCoordinates(tx_id=ENST00000650287, five_utr=FiveUTRCoordinates(regions=1 regions: (chr8, 19939252, 19939440, +)))]


To proceed next, each line of the variants file will be read and a line in the output CSV file will be witten with its corresponding ID and features results.

In [52]:
import os
import csv

from utrfx.model import TxperGene
from utrfx.genome import GRCh38, VariantCoordinates, GenomeBuild, Strand
from utrfx.util import fetch_cdna_from_ensembl, get_five_prime_sequence, uorf_extractor
from utrfx.uorf import gc_content, gc_content_n_bases_downstream, intercistronic_distance, cap_five_to_uorf_distance, kozak_sequence_strength  
from utrfx.variant_util import prepare_alt_seq, VCFfile

json_fpath = os.path.join("..", "..", "tests", "data", "Ensembl_transcript_per_gene_dictionary.json")
output_fpath = os.path.join("..", "..", "tests", "data", "HR_gene_results.tsv")

# Define csv header
max_uorfs = 4 
header = ["tx_id", "NuORFs"]
for i in range(max_uorfs):
    header += [f"{i}uORF_length", f"{i}Is_ouORF", f"{i}GC", f"{i}GCplus", f"{i}ID", f"{i}Cap_distance", f"{i}Kozak"]

# Create CSV file
with open(output_fpath, "w", newline='') as file:
    writer = csv.writer(file, delimiter='\t')
    writer.writerow(header)

# Iterate through each line
gene = None
tx_id = None
tx_id_clean = None
five_utr_cdna_sequence = None
contig = None
variants_id_list = []
for _, row in variants_df.iterrows():
    # For the the first variant of a gene, two results lines are obtained: one for the canonical transcript
    # and another for the first variant itself
    if gene is None or row["gene_symbol"] != gene:
        variants_id_list = []
        gene = row["gene_symbol"]
        tx_id = TxperGene(fpath=json_fpath).ensembl_transcript(gene_symbol=row["gene_symbol"])
        contig = GenomeBuild.contig_by_name(GRCh38, name= str(row["chr"]))
        
        # Our GTF file does not inform about the transcript ID version (usually indicated as a decimal number),
        # therefore this extra step is needed
        # You must check your file to make sure
        tx_id_clean = tx_id.split(".")[0]

        # Search the transcript within the ones in the GTF file
        for tx in transcripts:
            if tx.tx_id == tx_id_clean:
                tx_found = tx
                break
        
        assert tx_found is not None, "Transcript not found"

        # Get the 5'UTR cDNA sequence
        tx_cdna_sequence = fetch_cdna_from_ensembl(transcript_id=tx_id_clean)
        five_utr_cdna_sequence = get_five_prime_sequence(cdna_sequence=tx_cdna_sequence, five_utrs=tx_found.five_utr)

        # Create row that will keep the results and will be added to the final file
        results_tx = []

        # Add tx id to the results row
        tx_id_tsv = tx_id_clean + "-" + str(row["chr"])
        results_tx.append(str(tx_id_tsv))

        # Extract the Ensembl canonical transcript's uORFs
        uorfs = uorf_extractor(five_utr=tx_found.five_utr, five_sequence=five_utr_cdna_sequence)

        # Add number of uORFs to row
        results_tx.append(len(uorfs))

        # Add each feature
        for uorf in uorfs:
            results_tx.append(uorf.uorf.end - uorf.uorf.start)
            results_tx.append(uorf.ouorf)
            results_tx.append(gc_content(five_sequence=five_utr_cdna_sequence, uorf=uorf))
            results_tx.append(gc_content_n_bases_downstream(five_sequence=five_utr_cdna_sequence, uorf=uorf, bases=10))
            results_tx.append(intercistronic_distance(five_sequence=five_utr_cdna_sequence, uorf=uorf))
            results_tx.append(cap_five_to_uorf_distance(uorf=uorf))
            results_tx.append(kozak_sequence_strength(five_sequence=five_utr_cdna_sequence, uorf=uorf))      

        # If less uORFs than 4, fill with nan
        while len(results_tx) < len(header):
            results_tx.append(float("nan"))      

        # Write results in csv
        with open(output_fpath, "a", newline='') as file:
            writer = csv.writer(file, delimiter='\t')
            writer.writerow(results_tx)

    # Create the variant as a VariantCoordinates instance
    variant = VariantCoordinates.from_vcf_literal(contig, row["start"], row["ref"], row["alt"])

    # Define a variant id
    variant_tx_id = tx_id_clean + "-" + str(row["chr"]) + "-" + str(row["start"]) + "-" + row["ref"] + "-" + row["alt"]
    variants_id_list.append(variant_tx_id)

    # Repeat the process done with the canonical transcript
    variant_results = []
    variant_results.append(str(variant_tx_id))

    variant_five_utr_sequence = prepare_alt_seq(variant, five_utr_cdna_sequence, tx_found.five_utr)

    variant_uorfs = uorf_extractor(five_utr=tx_found.five_utr, five_sequence=variant_five_utr_sequence)
    variant_results.append(len(variant_uorfs))
    for uorf in variant_uorfs:
        variant_results.append(uorf.uorf.end - uorf.uorf.start)
        variant_results.append(uorf.ouorf)
        variant_results.append(gc_content(five_sequence=variant_five_utr_sequence, uorf=uorf))
        variant_results.append(gc_content_n_bases_downstream(five_sequence=variant_five_utr_sequence, uorf=uorf, bases=10))
        variant_results.append(intercistronic_distance(five_sequence=variant_five_utr_sequence, uorf=uorf))
        variant_results.append(cap_five_to_uorf_distance(uorf=uorf))
        variant_results.append(kozak_sequence_strength(five_sequence=variant_five_utr_sequence, uorf=uorf))
    
    while len(variant_results) < len(header):
        variant_results.append(float("nan"))

    with open(output_fpath, "a", newline='') as file:
        writer = csv.writer(file, delimiter='\t')
        writer.writerow(variant_results)

# Parse the VCF file with the GnomAD variants
with VCFfile(vcf_fpath) as vcf_file:
    for region in tx_found.five_utr.regions:
        vcf_variants = vcf_file.retrieve_variants_of_region(contig=contig, start=region.start_on_strand(Strand.POSITIVE), end=region.end_on_strand(Strand.POSITIVE))
        if vcf_variants:
            # Define a variant id
            for variant in vcf_variants:
                vcf_variant_tx_id = tx_id_clean + "-" + variant.chrom + "-" + str(variant.start) + "-" + variant.ref + "-" + variant.alt
                if vcf_variant_tx_id not in variants_id_list:
                    # Create the variant as a VariantCoordinates instance
                    vcf_variant = VariantCoordinates.from_vcf_literal(contig, variant.start, variant.ref, variant.alt)

                    # Repeat the process done with the canonical transcript
                    vcf_variant_results = []
                    vcf_variant_results.append(str(vcf_variant_tx_id))

                    variant_five_utr_sequence = prepare_alt_seq(vcf_variant, five_utr_cdna_sequence, tx_found.five_utr)

                    vcf_variant_uorfs = uorf_extractor(five_utr=tx_found.five_utr, five_sequence=variant_five_utr_sequence)
                    vcf_variant_results.append(len(vcf_variant_uorfs))
                    for uorf in vcf_variant_uorfs:
                        vcf_variant_results.append(uorf.uorf.end - uorf.uorf.start)
                        vcf_variant_results.append(uorf.ouorf)
                        vcf_variant_results.append(gc_content(five_sequence=variant_five_utr_sequence, uorf=uorf))
                        vcf_variant_results.append(gc_content_n_bases_downstream(five_sequence=variant_five_utr_sequence, uorf=uorf, bases=10))
                        vcf_variant_results.append(intercistronic_distance(five_sequence=variant_five_utr_sequence, uorf=uorf))
                        vcf_variant_results.append(cap_five_to_uorf_distance(uorf=uorf))
                        vcf_variant_results.append(kozak_sequence_strength(five_sequence=variant_five_utr_sequence, uorf=uorf))
                    
                    while len(vcf_variant_results) < len(header):
                        vcf_variant_results.append(float("nan"))

                    with open(output_fpath, "a", newline='') as file:
                        writer = csv.writer(file, delimiter='\t')
                        writer.writerow(vcf_variant_results)

Finally, you should get a results file similar to this:

In [53]:
import csv

with open(output_fpath, mode="r", encoding="utf-8") as file:
    reader = csv.reader(file)
    
    for row in reader:
        print(row)

['tx_id\tNuORFs\t0uORF_length\t0Is_ouORF\t0GC\t0GCplus\t0ID\t0Cap_distance\t0Kozak\t1uORF_length\t1Is_ouORF\t1GC\t1GCplus\t1ID\t1Cap_distance\t1Kozak\t2uORF_length\t2Is_ouORF\t2GC\t2GCplus\t2ID\t2Cap_distance\t2Kozak\t3uORF_length\t3Is_ouORF\t3GC\t3GCplus\t3ID\t3Cap_distance\t3Kozak']
['ENST00000381418-8\t4\t51\tFalse\t0.6274509803921569\t0.9\t556\t16\t2\t105\tFalse\t0.7047619047619048\t0.9\t216\t302\t1\t66\tFalse\t0.8181818181818182\t0.7\t47\t510\t1\t17\tTrue\t0.5882352941176471\t0\t0\t606\t0']
['ENST00000381418-8-22130605-T-C\t4\t51\tFalse\t0.6274509803921569\t0.9\t556\t16\t2\t321\tTrue\t0.7632398753894081\t0\t0\t302\t1\t66\tFalse\t0.8181818181818182\t0.7\t47\t510\t1\t17\tTrue\t0.5882352941176471\t0\t0\t606\t0']
['ENST00000381418-8-22130606-A-G\t4\t51\tFalse\t0.6274509803921569\t0.9\t556\t16\t2\t321\tTrue\t0.7632398753894081\t0\t0\t302\t1\t66\tFalse\t0.8181818181818182\t0.7\t47\t510\t1\t17\tTrue\t0.5882352941176471\t0\t0\t606\t0']
['ENST00000381418-8-22130627-C-G\t4\t51\tFalse\t0.627